# Amazon Bedrock AgentCore + CrewAI - Code Validation

This notebook validates all the code from blog #8: **Deploying CrewAI Multi-Agent Teams with Amazon Bedrock AgentCore: From Development to Production**

## 📖 Introduction

Amazon Bedrock AgentCore is AWS's secure, serverless runtime for deploying AI agents at scale. Unlike traditional containerized deployments, AgentCore provides:

- **Serverless Runtime**: No infrastructure management required
- **Session Isolation**: Each user session runs in isolated environments
- **Extended Execution**: Support for both real-time interactions and long-running workloads
- **Built-in Features**: Authentication, observability, memory, and tools
- **Consumption-based Pricing**: Pay only for what you use

CrewAI enables multi-agent collaboration where specialized agents work together to solve complex problems, similar to human teams but at machine speed.

## 🎯 Key Points Validated

1. **AgentCore Integration**: Using `BedrockAgentCoreApp` with `@app.entrypoint` pattern
2. **CrewAI Multi-Agent Setup**: Research, Analysis, and Writing agents working collaboratively
3. **Bedrock LLM Integration**: Using CrewAI's LLM wrapper for Bedrock models
4. **Custom Tools**: Knowledge search tools that agents can use
5. **Production Patterns**: Error handling, rate limiting, and deployment concepts
6. **Local Development**: Testing workflows before AgentCore deployment

## 🚀 Quick Summary

**What AgentCore Provides:**
- Serverless deployment for AI agents (framework agnostic)
- Built-in security, scaling, and observability
- Integration with any agent framework (CrewAI, LangGraph, Strands Agents, Google ADK, OpenAI Agents, etc.)

**What CrewAI Enables:**
- Multi-agent collaboration with specialized roles
- Sequential task execution with context sharing
- Tool integration for enhanced capabilities

**Deployment Flow (AgentCore CLI):**
1. Develop locally with CrewAI + `agentcore dev`
2. Package with AgentCore SDK (`@app.entrypoint` pattern)
3. Deploy with `agentcore deploy` (uses AWS CDK under the hood)
4. Invoke with `agentcore invoke` or boto3 SDK

## 📋 Prerequisites
- AWS credentials configured
- Python 3.10+
- Node.js 20+ (for AgentCore CLI)
- Bedrock model access enabled
- Required packages (installed automatically)

In [11]:
# Basic imports
import boto3
import json
from typing import Dict, Any

print("✅ Imports successful")
print(f"Boto3 version: {boto3.__version__}")

✅ Imports successful
Boto3 version: 1.42.97


In [12]:
# AWS SSO login and connection test
import os
import subprocess

try:
    print("🔐 Attempting AWS SSO login for default profile...")
    result = subprocess.run(['aws', 'sso', 'login', '--profile', 'default'],
                          capture_output=True, text=True, timeout=60)

    if result.returncode != 0:
        print(f"⚠️ SSO login output: {result.stderr}")

    session = boto3.Session(profile_name='default')
    credentials = session.get_credentials()

    os.environ['AWS_ACCESS_KEY_ID'] = credentials.access_key
    os.environ['AWS_SECRET_ACCESS_KEY'] = credentials.secret_key
    if credentials.token:
        os.environ['AWS_SESSION_TOKEN'] = credentials.token
    os.environ['AWS_DEFAULT_REGION'] = session.region_name or 'us-east-1'

    sts = session.client('sts')
    identity = sts.get_caller_identity()
    print(f"✅ AWS connected with default profile: {identity['Account']}")
    print(f"Region: {os.environ['AWS_DEFAULT_REGION']}")
    aws_ready = True

except subprocess.TimeoutExpired:
    print("⚠️ SSO login timed out - please run 'aws sso login --profile default' manually")
    aws_ready = False
except Exception as e:
    print(f"❌ AWS failed: {e}")
    print("💡 Try running: aws sso login --profile default")
    aws_ready = False

🔐 Attempting AWS SSO login for default profile...
✅ AWS connected with default profile: 123456789012
Region: us-east-1


In [13]:
# CrewAI imports (Python 3.10+)
try:
    from crewai import Agent, Crew, Process, Task
    from crewai.tools import BaseTool
    print("✅ CrewAI imported successfully")
    crewai_ready = True
except ImportError as e:
    print(f"❌ CrewAI failed: {e}")
    print("💡 Make sure you're using the 'CrewAI Python 3.10' kernel")
    crewai_ready = False

✅ CrewAI imported successfully


In [14]:
# Custom Knowledge Tool
if crewai_ready:
    class KnowledgeSearchTool(BaseTool):
        name: str = "knowledge_search"
        description: str = "Search AWS knowledge"

        def _run(self, query: str) -> str:
            kb = {
                "s3": "Object storage with 99.999999999% durability",
                "lambda": "Serverless compute with automatic scaling",
                "agentcore": "Secure serverless runtime for AI agents"
            }
            results = [f"{k.upper()}: {v}" for k, v in kb.items()
                      if any(w.lower() in v.lower() for w in query.split())]
            return "\n".join(results) or "No results found"

    tool = KnowledgeSearchTool()
    for query in ["serverless", "storage", "compute"]:
        print(f"🔍 Query '{query}': {tool._run(query)}")
    print("\n✅ Knowledge tool working")

🔍 Query 'serverless': LAMBDA: Serverless compute with automatic scaling
AGENTCORE: Secure serverless runtime for AI agents
🔍 Query 'storage': S3: Object storage with 99.999999999% durability
🔍 Query 'compute': LAMBDA: Serverless compute with automatic scaling

✅ Knowledge tool working


In [15]:
# Bedrock LLM for CrewAI
# NOTE: CrewAI uses litellm internally - no need to import langchain_aws
if aws_ready:
    try:
        from crewai import LLM

        llm = LLM(
            model="bedrock/us.anthropic.claude-haiku-4-5-20251001-v1:0",
            temperature=0.1
        )
        print("✅ CrewAI Bedrock LLM ready")

    except Exception as e:
        llm = None
        print(f"❌ Bedrock failed: {e}")
else:
    llm = None
    print("⚠️ No AWS - skipping Bedrock")

✅ CrewAI Bedrock LLM ready


In [16]:
# Create agents
if crewai_ready and llm:
    researcher = Agent(
        role="AWS Researcher",
        goal="Find AWS service info",
        backstory="Expert researcher",
        tools=[tool] if 'tool' in locals() else [],
        llm=llm
    )

    analyst = Agent(
        role="Technical Analyst",
        goal="Analyze findings",
        backstory="Senior analyst",
        llm=llm
    )

    print("✅ Agents created")
    agents_ready = True
else:
    print("⚠️ Skipping agents - dependencies missing")
    agents_ready = False

✅ Agents created


In [17]:
# Test simple workflow with rate limiting handling
if agents_ready:
    simple_task = Task(
        description="Briefly explain what AWS Lambda is",
        agent=researcher,
        expected_output="Short explanation of AWS Lambda"
    )

    crew = Crew(
        agents=[researcher],
        tasks=[simple_task],
        process=Process.sequential,
        verbose=False
    )

    print("✅ Simple workflow created")

    try:
        import time
        print("⏳ Waiting 5 seconds to avoid rate limits...")
        time.sleep(5)

        result = crew.kickoff()
        print(f"✅ Executed: {str(result)[:100]}...")
    except Exception as e:
        if "rate limit" in str(e).lower() or "too many requests" in str(e).lower():
            print("⚠️ Rate limited - try again in a few minutes")
        else:
            print(f"⚠️ Execution failed: {e}")
else:
    print("⚠️ Skipping workflow test")

✅ Simple workflow created
⏳ Waiting 5 seconds to avoid rate limits...
✅ Executed: Based on my existing knowledge of AWS services, here is a brief explanation of AWS Lambda:

---

## ...


# 🚀 Production Deployment Guide - Amazon Bedrock AgentCore

## Overview

Amazon Bedrock AgentCore provides a comprehensive deployment platform for AI agents with enterprise-grade features:

- **Serverless Runtime**: No infrastructure management required
- **Automatic Scaling**: Handles traffic spikes without configuration
- **Built-in Security**: IAM integration, session isolation, and secure execution
- **Observability**: CloudWatch integration for monitoring and debugging
- **Multi-Framework Support**: Works with CrewAI, LangGraph, Strands Agents, Google ADK, OpenAI Agents, and custom frameworks

## Supported Regions

AgentCore Runtime is available in **15 AWS regions**:
us-east-1, us-east-2, us-west-2, eu-central-1, eu-west-1, eu-west-2, eu-west-3, eu-north-1,
ap-south-1, ap-southeast-1, ap-southeast-2, ap-northeast-1, ap-northeast-2, ca-central-1, sa-east-1

## Prerequisites

### 1. AWS Account Setup
```bash
aws sts get-caller-identity
aws configure --profile your-profile
```

### 2. Node.js 20+ (required for AgentCore CLI)
```bash
node --version  # Must be 20+
```

### 3. Model Access
Enable Bedrock model access in the AWS Console:
```
Navigate to Amazon Bedrock Console > Model Access
Enable: Anthropic Claude Sonnet 4.6 (recommended)
```

## Step-by-Step Deployment Process

### Step 1: Install AgentCore CLI

```bash
# Install the AgentCore CLI (npm package)
npm install -g @aws/agentcore

# Verify installation
agentcore --version

# Install Python SDK
pip install bedrock-agentcore crewai
```

### Step 2: Create Agent Project

```bash
# Interactive wizard
agentcore create

# Or non-interactive
agentcore create --name CrewAIAgent --framework Strands --model-provider Bedrock --memory none
```

This generates the project structure:
```
CrewAIAgent/
├── agentcore/
│   ├── agentcore.json      # Project and resource configuration
│   ├── aws-targets.json    # Deployment target (account and region)
│   └── cdk/                # CDK infrastructure (auto-managed)
└── app/
    └── CrewAIAgent/
        ├── main.py         # Agent entrypoint (replace with CrewAI code)
        └── pyproject.toml  # Python dependencies
```

### Step 3: Create Production Agent File

Replace `app/CrewAIAgent/main.py` with your CrewAI agent code:

```python
from bedrock_agentcore.runtime import BedrockAgentCoreApp
from crewai import Agent, Crew, Process, Task, LLM
from crewai.tools import BaseTool
import logging

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

app = BedrockAgentCoreApp()

class KnowledgeSearchTool(BaseTool):
    name: str = "knowledge_search"
    description: str = "Search AWS documentation and technical knowledge"
    def _run(self, query: str) -> str:
        return f"Knowledge search results for: {query}"

llm = LLM(
    model="bedrock/us.anthropic.claude-haiku-4-5-20251001-v1:0",
    temperature=0.1
)

researcher = Agent(
    role="AWS Research Specialist",
    goal="Find comprehensive AWS service information",
    backstory="Expert researcher with deep AWS knowledge",
    tools=[KnowledgeSearchTool()],
    llm=llm, verbose=False
)
analyst = Agent(
    role="Technical Analyst",
    goal="Analyze research findings",
    backstory="Senior analyst specializing in cloud architecture",
    llm=llm, verbose=False
)
writer = Agent(
    role="Technical Writer",
    goal="Create clear responses for developers",
    backstory="Expert technical writer",
    llm=llm, verbose=False
)

@app.entrypoint
def crewai_handler(payload, context):
    try:
        user_input = payload.get("prompt", "How can I help you?")
        research_task = Task(description=f"Research: {user_input}",
                            agent=researcher, expected_output="Research findings")
        analysis_task = Task(description="Analyze the research findings",
                            agent=analyst, expected_output="Analysis results",
                            context=[research_task])
        writing_task = Task(description="Write a comprehensive response",
                           agent=writer, expected_output="Final response",
                           context=[research_task, analysis_task])
        crew = Crew(
            agents=[researcher, analyst, writer],
            tasks=[research_task, analysis_task, writing_task],
            process=Process.sequential, verbose=False
        )
        result = crew.kickoff()
        return {"result": result.raw, "status": "success",
                "session_id": context.session_id}
    except Exception as e:
        logger.error(f"Error: {str(e)}")
        return {"result": f"Error: {str(e)}", "status": "error"}

if __name__ == "__main__":
    app.run()
```

Update `pyproject.toml` dependencies:
```toml
[project]
dependencies = [
    "bedrock-agentcore",
    "crewai",
]
```

### Step 4: Local Testing

```bash
cd CrewAIAgent

# Start local dev server (opens agent inspector in browser)
agentcore dev

# Or non-interactive with logs
agentcore dev --no-browser --logs

# Test in another terminal
curl -X POST http://localhost:8080/invocations \
  -H "Content-Type: application/json" \
  -d '{"prompt": "What AWS services are best for serverless applications?"}'

# Or use the CLI
agentcore dev "What AWS services are best for serverless?"
```

### Step 5: Deploy to AgentCore Runtime

```bash
# Preview what will change
agentcore deploy --plan

# Deploy (uses AWS CDK under the hood)
agentcore deploy

# Check status
agentcore status
```

**Deployment Process (what happens under the hood):**
1. Packages your code (CodeZip archive or Docker container)
2. Uses AWS CDK to synthesize and provision CloudFormation resources
3. Creates IAM roles and AgentCore Runtime endpoint
4. Configures CloudWatch logging and observability

### Step 6: Test Deployed Agent

```bash
# Quick test using CLI
agentcore invoke "Explain AWS Lambda benefits"

# With streaming
agentcore invoke --prompt "Explain AWS Lambda benefits" --stream

# Maintain session context
agentcore invoke --session-id my-session "Tell me more about that"
```

## Production Invocation with boto3

```python
import boto3
import json
import uuid

def invoke_crewai_agent(agent_arn: str, prompt: str, session_id: str = None):
    client = boto3.client('bedrock-agentcore')
    payload = json.dumps({"prompt": prompt}).encode()
    session_id = session_id or str(uuid.uuid4())

    response = client.invoke_agent_runtime(
        agentRuntimeArn=agent_arn,
        runtimeSessionId=session_id,  # Required parameter
        payload=payload,
        qualifier="DEFAULT"
    )

    # Handle response based on content type
    content_type = response.get("contentType", "")
    if "text/event-stream" in content_type:
        content = []
        for line in response["response"].iter_lines(chunk_size=10):
            if line:
                line = line.decode("utf-8")
                if line.startswith("data: "):
                    content.append(line[6:])
        return "\n".join(content)
    elif content_type == "application/json":
        content = []
        for chunk in response.get("response", []):
            content.append(chunk.decode('utf-8'))
        return json.loads(''.join(content))
    else:
        return response

# Usage
result = invoke_crewai_agent(
    "arn:aws:bedrock-agentcore:us-east-1:123456789012:agent-runtime/abc123",
    "What's the best architecture for a data lake?"
)
print(result)
```

## Observability

```bash
# Stream agent logs
agentcore logs

# Filter logs
agentcore logs --since 30m --level error
agentcore logs --query "timeout"

# List recent traces
agentcore traces list

# Get a specific trace
agentcore traces get <trace-id>
```

## Common Issues and Solutions

1. **Permission Denied**: Verify IAM permissions and execution role
2. **Model Access Denied**: Enable Bedrock model access in console
3. **Rate Limiting**: Implement exponential backoff in agent code
4. **CDK Deployment Errors**: Ensure CDK is bootstrapped (`cdk bootstrap`)
5. **Port 8080 in use**: Use `agentcore dev --port 8081`

## Cleanup

```bash
# Remove all resources from config
agentcore remove all

# Deploy empty state to tear down AWS resources
agentcore deploy
```

## Summary

Amazon Bedrock AgentCore provides enterprise-grade deployment for CrewAI multi-agent systems with:

- **Zero Infrastructure Management**: Fully serverless deployment
- **Automatic Scaling**: Handles traffic without configuration
- **Built-in Security**: IAM integration and session isolation
- **Production Monitoring**: CloudWatch integration and observability
- **Cost Optimization**: Pay-per-use consumption model

This enables you to focus on building intelligent agent workflows while AWS handles all operational complexity.

## 🛠️ Step 1: Install AgentCore CLI & SDK

In [18]:
# Step 1: Verify AgentCore SDK & CLI
import importlib

for pkg, mod in [("bedrock-agentcore", "bedrock_agentcore"), ("crewai", "crewai"), ("boto3", "boto3")]:
    try:
        m = importlib.import_module(mod)
        ver = getattr(m, "__version__", "OK")
        print(f"✅ {pkg}: {ver}")
    except ImportError:
        print(f"❌ {pkg} not found — run: uv pip install {pkg}")

import subprocess
try:
    r = subprocess.run(["agentcore", "--version"], capture_output=True, text=True)
    print(f"✅ AgentCore CLI: {r.stdout.strip()}" if r.returncode == 0 else "⚠️ AgentCore CLI not found")
except FileNotFoundError:
    print("⚠️ AgentCore CLI not in PATH — install with: npm install -g @aws/agentcore")


✅ bedrock-agentcore: OK
✅ crewai: 1.14.3
✅ boto3: 1.42.97
✅ AgentCore CLI: 0.11.0



## 📝 Step 2: Create Production Agent File

In [19]:
# Step 2: Create production-ready CrewAI agent file
agent_code = '''from bedrock_agentcore.runtime import BedrockAgentCoreApp
from crewai import Agent, Crew, Process, Task, LLM
from crewai.tools import BaseTool
import logging

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

app = BedrockAgentCoreApp()

class KnowledgeSearchTool(BaseTool):
    name: str = "knowledge_search"
    description: str = "Search AWS documentation and technical knowledge"
    def _run(self, query: str) -> str:
        kb = {
            "s3": "Object storage with 99.999999999% durability",
            "lambda": "Serverless compute with automatic scaling",
            "agentcore": "Secure serverless runtime for AI agents"
        }
        results = [f"{k.upper()}: {v}" for k, v in kb.items()
                  if any(w.lower() in v.lower() for w in query.split())]
        return "\n".join(results) or "No results found"

llm = LLM(
    model="bedrock/us.anthropic.claude-haiku-4-5-20251001-v1:0",
    temperature=0.1
)

researcher = Agent(
    role="AWS Research Specialist",
    goal="Find comprehensive AWS service information",
    backstory="Expert researcher with deep AWS knowledge",
    tools=[KnowledgeSearchTool()],
    llm=llm, verbose=False
)

@app.entrypoint
def crewai_handler(payload, context):
    """Main AgentCore entrypoint for CrewAI workflow"""
    try:
        user_input = payload.get("prompt", "How can I help you?")
        logger.info(f"Processing: {user_input}")

        research_task = Task(
            description=f"Research: {user_input}",
            agent=researcher,
            expected_output="Research findings"
        )

        crew = Crew(
            agents=[researcher],
            tasks=[research_task],
            process=Process.sequential,
            verbose=False
        )

        result = crew.kickoff()
        return {
            "result": result.raw,
            "status": "success",
            "session_id": context.session_id
        }
    except Exception as e:
        logger.error(f"Error: {str(e)}")
        return {"result": f"Error: {str(e)}", "status": "error"}

if __name__ == "__main__":
    app.run()
'''

with open('crewai_agent.py', 'w') as f:
    f.write(agent_code)

print("✅ Created crewai_agent.py")
print("\n📁 Files ready for AgentCore deployment:")
print("  - crewai_agent.py (main agent file)")

✅ Created crewai_agent.py

📁 Files ready for AgentCore deployment:
  - crewai_agent.py (main agent file)


## 🧪 Step 3: Local Testing

In [10]:
# Step 3: Test the agent locally before deployment
import subprocess
import time
import requests
import json
import os

def test_local_agent():
    """Test the CrewAI agent locally"""
    print("🚀 Starting local agent server...")
    try:
        process = subprocess.Popen(
            ['python', 'crewai_agent.py'],
            stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True
        )
        print("⏳ Waiting for server to start...")
        time.sleep(15)

        try:
            response = requests.post(
                'http://localhost:8080/invocations',
                json={"prompt": "What is AWS Lambda?"},
                timeout=30
            )
            if response.status_code == 200:
                result = response.json()
                print("✅ Local test successful!")
                print(f"📝 Response: {result.get('result', 'No result')[:100]}...")
            else:
                print(f"❌ Test failed: {response.status_code}")
        except requests.exceptions.RequestException as e:
            print(f"❌ Request failed: {e}")

        print("\n🛑 Stopping local server...")
        process.terminate()
        process.wait(timeout=5)
        print("✅ Local server stopped")
    except Exception as e:
        print(f"❌ Local test failed: {e}")
        if 'process' in locals():
            process.terminate()

if os.path.exists('crewai_agent.py'):
    print("📁 Found crewai_agent.py")
    test_local_agent()

    print("\n🎯 Next Steps for AgentCore Deployment:")
    print("1. npm install -g @aws/agentcore")
    print("2. agentcore create --name CrewAIAgent --defaults")
    print("3. # Copy crewai_agent.py to app/CrewAIAgent/main.py")
    print("4. agentcore dev")
    print("5. agentcore deploy")
    print('6. agentcore invoke "What is AWS Lambda?"')
else:
    print("❌ crewai_agent.py not found - run Step 2 first")

📁 Found crewai_agent.py
🚀 Starting local agent server...
❌ Local test failed: [Errno 2] No such file or directory: 'python'

🎯 Next Steps for AgentCore Deployment:
1. npm install -g @aws/agentcore
2. agentcore create --name CrewAIAgent --defaults
3. # Copy crewai_agent.py to app/CrewAIAgent/main.py
4. agentcore dev
5. agentcore deploy
6. agentcore invoke "What is AWS Lambda?"


## 🚀 Step 4: Deploy to Amazon Bedrock AgentCore Runtime

This section creates an AgentCore project, injects our CrewAI agent code, and deploys to AWS.

**What will be created in your AWS account:**
- IAM roles for AgentCore Runtime
- AgentCore Runtime endpoint
- CloudWatch log groups

All resources are removable with the cleanup cell at the end.

In [21]:
# Step 4a: Create AgentCore project and inject CrewAI agent code
import subprocess, os, shutil, json

PROJECT = "CrewAIAgent"
PROJECT_DIR = os.path.join(os.getcwd(), PROJECT)

# Clean previous project if exists
if os.path.exists(PROJECT_DIR):
    shutil.rmtree(PROJECT_DIR)
    print(f"🗑️ Removed existing {PROJECT}/")

# Create project non-interactively
print("📦 Creating AgentCore project...")
r = subprocess.run(
    ["agentcore", "create", "--name", PROJECT, "--defaults"],
    capture_output=True, text=True, cwd=os.getcwd()
)
if r.returncode != 0:
    print(f"❌ Failed: {r.stderr}")
else:
    print("✅ Project created")

    # Copy our CrewAI agent code into the project
    main_py = os.path.join(PROJECT_DIR, "app", PROJECT, "main.py")
    shutil.copy("crewai_agent.py", main_py)
    print(f"✅ Copied crewai_agent.py → {main_py}")

    # Update pyproject.toml to add crewai dependency
    pyproject = os.path.join(PROJECT_DIR, "app", PROJECT, "pyproject.toml")
    with open(pyproject, "r") as f:
        content = f.read()
    if "crewai" not in content:
        content = content.replace(
            'dependencies = [',
            'dependencies = [\n    "crewai",'
        )
        with open(pyproject, "w") as f:
            f.write(content)
        print("✅ Added crewai to pyproject.toml")

    # Show project structure
    print(f"\n📁 Project structure:")
    for root, dirs, files in os.walk(PROJECT_DIR):
        dirs[:] = [d for d in dirs if d not in ('.venv', 'node_modules', 'cdk', '.cli')]
        level = root.replace(PROJECT_DIR, '').count(os.sep)
        indent = '  ' * level
        print(f"{indent}{os.path.basename(root)}/")
        for f in files:
            print(f"{indent}  {f}")

    # Configure deployment target (account + region)
    import boto3
    session = boto3.Session(profile_name='default')
    account = session.client('sts').get_caller_identity()['Account']
    region = session.region_name or 'us-east-1'
    targets_file = os.path.join(PROJECT_DIR, 'agentcore', 'aws-targets.json')
    with open(targets_file, 'w') as f:
        json.dump([{'name': 'default', 'account': account, 'region': region}], f, indent=2)
    print(f'✅ Configured target: account={account}, region={region}')


🗑️ Removed existing CrewAIAgent/
📦 Creating AgentCore project...
✅ Project created
✅ Copied crewai_agent.py → /Users/ccortez/dev/breakingthecloud/blog-v2/notebooks/8_blog/8_2_new/CrewAIAgent/app/CrewAIAgent/main.py
✅ Added crewai to pyproject.toml

📁 Project structure:
CrewAIAgent/
  README.md
  AGENTS.md
  app/
    CrewAIAgent/
      uv.lock
      pyproject.toml
      README.md
      .gitignore
      main.py
      mcp_client/
        client.py
        __init__.py
      model/
        __init__.py
        load.py
  agentcore/
    .env.local
    agentcore.json
    aws-targets.json
    .gitignore
    .llm-context/
      agentcore.ts
      README.md
      aws-targets.ts


In [28]:
# Setup AWS credentials for agentcore CLI
import boto3, os
session = boto3.Session(profile_name='default')
creds = session.get_credentials()
os.environ['AWS_ACCESS_KEY_ID'] = creds.access_key
os.environ['AWS_SECRET_ACCESS_KEY'] = creds.secret_key
if creds.token: os.environ['AWS_SESSION_TOKEN'] = creds.token
os.environ['AWS_REGION'] = session.region_name or 'us-east-1'
os.environ['AWS_DEFAULT_REGION'] = os.environ['AWS_REGION']

# Step 4b: Deploy to AgentCore Runtime
# ⚠️ This creates AWS resources in your account (IAM roles, AgentCore Runtime, CloudWatch logs)
import subprocess, os

PROJECT_DIR = os.path.join(os.getcwd(), "CrewAIAgent")

print("🚀 Deploying to AgentCore Runtime...")
print("⏳ First deploy takes a few minutes (CDK bootstrap + build)...\n")

r = subprocess.run(
    ["agentcore", "deploy", "--yes", "--verbose"],
    capture_output=True, text=True, cwd=PROJECT_DIR,
    timeout=600  # 10 min timeout
)

print(r.stdout[-2000:] if len(r.stdout) > 2000 else r.stdout)
if r.returncode != 0:
    print(f"\n❌ Deploy failed:\n{r.stderr[-1000:]}")
else:
    print("\n✅ Deploy successful!")

# Get status
s = subprocess.run(["agentcore", "status", "--json"],
    capture_output=True, text=True, cwd=PROJECT_DIR)
if s.returncode == 0:
    print(f"\n📋 Status:\n{s.stdout[:1000]}")

# Add Marketplace permissions to execution role (required for Anthropic models)
if r.returncode == 0:
    import re, json as j
    role_match = re.search(r'RoleArnOutput\S*: (arn:aws:iam::\S+)', r.stdout)
    if role_match:
        role_name = role_match.group(1).split('/')[-1]
        iam = boto3.client('iam')
        iam.put_role_policy(
            RoleName=role_name,
            PolicyName='MarketplaceModelAccess',
            PolicyDocument=j.dumps({
                'Version': '2012-10-17',
                'Statement': [{
                    'Effect': 'Allow',
                    'Action': ['aws-marketplace:ViewSubscriptions', 'aws-marketplace:Subscribe', 'aws-marketplace:Unsubscribe'],
                    'Resource': '*'
                }]
            })
        )
        print(f'✅ Marketplace permissions added to {role_name}')


🚀 Deploying to AgentCore Runtime...
⏳ First deploy takes a few minutes (CDK bootstrap + build)...

ploy to AWS...
⠧ Deploy to AWS...
⠇ Deploy to AWS...
⠏ Deploy to AWS...
⠋ Deploy to AWS...
 ✅  AgentCore-CrewAIAgent-default (no changes)
Outputs:
AgentCore-CrewAIAgent-default.ApplicationAgentCrewAIAgentRoleArnOutput9076F980 = arn:aws:iam::123456789012:role/AgentCore-CrewAIAgent-def-ApplicationAgentCrewAIAge-ExampleRoleId
AgentCore-CrewAIAgent-default.ApplicationAgentCrewAIAgentRuntimeArnOutputE9E7A9BB = arn:aws:bedrock-agentcore:us-east-1:123456789012:runtime/CrewAIAgent_CrewAIAgent-ExampleRuntimeId
AgentCore-CrewAIAgent-default.ApplicationAgentCrewAIAgentRuntimeIdOutputEDE18AD2 = CrewAIAgent_CrewAIAgent-ExampleRuntimeId
AgentCore-CrewAIAgent-default.StackNameOutput = AgentCore-CrewAIAgent-default
Stack ARN:
arn:aws:cloudformation:us-east-1:123456789012:stack/AgentCore-CrewAIAgent-default/a1b2c3d4-5678-90ab-cdef-example12345

✓ Deploy to AWS

⠋ Persist deployment state...
⠙ Persist depl

## 🧪 Step 5: Invoke Deployed Agent

In [32]:
# Setup AWS credentials for agentcore CLI
import boto3, os, json as j
session = boto3.Session(profile_name='default')
creds = session.get_credentials()
os.environ['AWS_ACCESS_KEY_ID'] = creds.access_key
os.environ['AWS_SECRET_ACCESS_KEY'] = creds.secret_key
if creds.token: os.environ['AWS_SESSION_TOKEN'] = creds.token
os.environ['AWS_REGION'] = session.region_name or 'us-east-1'
os.environ['AWS_DEFAULT_REGION'] = os.environ['AWS_REGION']

# Step 5: Invoke the deployed agent
import subprocess

PROJECT_DIR = os.path.join(os.getcwd(), "CrewAIAgent")

print("🤖 Invoking deployed agent...")
print("⏳ CrewAI workflow takes ~30-60s to complete...\n")

r = subprocess.run(
    ["agentcore", "invoke", "--prompt", "What is AWS Lambda? Answer in 3 sentences.", "--json"],
    capture_output=True, text=True, cwd=PROJECT_DIR,
    timeout=180
)

if r.returncode == 0:
    try:
        # Parse JSON and extract just the response
        data = j.loads(r.stdout.strip().split('\n')[-1])
        response = data.get('response', '')
        session_id = data.get('sessionId', '')
        print("✅ Response from AgentCore:\n")
        print(response)
        print(f"\n📋 Session: {session_id}")
    except Exception as e:
        print(f"Raw output:\n{r.stdout[-1000:]}")
else:
    print(f"❌ Invoke failed:\n{r.stderr[-500:]}")


🤖 Invoking deployed agent...
⏳ CrewAI workflow takes ~30-60s to complete...

Raw output:
default","response":"AWS Lambda is a serverless compute service that allows you to run code without provisioning or managing servers, automatically scaling based on demand and supporting event-driven execution models. It enables developers to execute functions in response to various triggers and events from AWS services, third-party applications, and custom applications, with built-in features like automatic scaling, monitoring through Lambda Insights, code organization through Layers, and direct HTTPS endpoints via Function URLs. Lambda operates on a pay-per-use pricing model where you only pay for the compute time consumed during function execution, making it a cost-effective solution for building scalable, event-driven applications and microservices architectures.","sessionId":"24b00ce5-5d53-4866-b695-c459af2b6840","logFilePath":"/Users/ccortez/dev/breakingthecloud/blog-v2/notebooks/8_blog/8_2_n

## 🧹 Step 6: Cleanup

⚠️ Run this cell to remove all AWS resources created by the deploy.

In [33]:
# Setup AWS credentials for agentcore CLI
import boto3, os
session = boto3.Session(profile_name='default')
creds = session.get_credentials()
os.environ['AWS_ACCESS_KEY_ID'] = creds.access_key
os.environ['AWS_SECRET_ACCESS_KEY'] = creds.secret_key
if creds.token: os.environ['AWS_SESSION_TOKEN'] = creds.token
os.environ['AWS_REGION'] = session.region_name or 'us-east-1'
os.environ['AWS_DEFAULT_REGION'] = os.environ['AWS_REGION']

# Step 6: Cleanup - Remove all AWS resources
# ⚠️ This will tear down the AgentCore Runtime and associated resources
import subprocess, os

PROJECT_DIR = os.path.join(os.getcwd(), "CrewAIAgent")

print("🧹 Removing AgentCore resources...")

# Remove all resources from config
r1 = subprocess.run(
    ["agentcore", "remove", "all", "--yes"],
    capture_output=True, text=True, cwd=PROJECT_DIR
)
print(r1.stdout)

# Deploy empty state to tear down AWS resources
print("⏳ Tearing down AWS resources...")
r2 = subprocess.run(
    ["agentcore", "deploy", "--yes"],
    capture_output=True, text=True, cwd=PROJECT_DIR,
    timeout=300
)
print(r2.stdout[-1000:] if len(r2.stdout) > 1000 else r2.stdout)

if r2.returncode == 0:
    print("\n✅ All AWS resources removed")
else:
    print(f"\n⚠️ Cleanup may be incomplete: {r2.stderr[-500:]}")


🧹 Removing AgentCore resources...
{"success":true,"message":"All schemas reset to empty state","note":"Your source code has not been modified. Run `agentcore deploy` to apply changes to AWS."}

⏳ Tearing down AWS resources...
k...
⠼ Tear down stack...
⠴ Tear down stack...
⠦ Tear down stack...
⠧ Tear down stack...
⠇ Tear down stack...
⠏ Tear down stack...
⠋ Tear down stack...
⠙ Tear down stack...
⠹ Tear down stack...
⠸ Tear down stack...
⠼ Tear down stack...
⠴ Tear down stack...
⠦ Tear down stack...
⠧ Tear down stack...
⠇ Tear down stack...
⠏ Tear down stack...
⠋ Tear down stack...
⠙ Tear down stack...
⠹ Tear down stack...
⠸ Tear down stack...
⠼ Tear down stack...
⠴ Tear down stack...
⠦ Tear down stack...
⠧ Tear down stack...
⠇ Tear down stack...
⠏ Tear down stack...
⠋ Tear down stack...
⠙ Tear down stack...
⠹ Tear down stack...
⠸ Tear down stack...
⠼ Tear down stack...
⠴ Tear down stack...
⠦ Tear down stack...
⠧ Tear down stack...
⠇ Tear down stack...
⠏ Tear down stack...
⠋ Tear down s